In [ ]:
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm  
import time

import json
import re

TARGET_MODEL = "qwen/qwen-2.5-72b-instruct" 

client = OpenAI(
    api_key="sk-or-", # OpenRouter API key — please don't leak it :) 
    base_url="https://openrouter.ai/api/v1"
)

# read the datasets
df_test = pd.read_excel("Project ME_validation 100.xlsx", sheet_name = 'Input_WoS_100')
df_validate = pd.read_excel("Project ME_validation 100.xlsx", sheet_name = 'Manual validation')

In [15]:

SYSTEM_PROMPT = """ You are an Ecosystem Service expert and a dedicated assistant designed to classify research articles (titles, keywords, abstracts given) based on the following instructions:
1. **Ecosystem Service Technology Analysis:**
Determine if the abstract describes a technological intervention that contributes to one or more of the following ecosystem services:
**Provisioning—Products obtained from ecosystems (Existing commercial market):**
- Biodiversity—The number of different species
- Food—Ingredients derived from wild and domesticated habitats
- Potable Water—Fresh water that is safe to consume
- Fuel—Materials used to generate energy
- Fibre/Hide/Wood—Materials used for clothing or construction
- Biochemicals—Molecules used in medicine
**Cultural—Benefits to quality of life and community (Existing commercial market):**
- Spiritual—Supporting the spiritual lives of people
- Recreation—Supporting the physical and mental health of people
- Aesthetic—The mental and physical health benefits of natural beauty
- Inspiration/Education—Art, music, literature, architecture, and engineering design
- Cultural Heritage—Value placed upon landscapes
- Cultural Identity—Societal identity regulated by the ecosystem (e.g., nomadic herding)
**Regulating—Benefits obtained by regulating ecosystem processes (Most amenable to technological replacement):**
- Atmospheric Regulation—Production and consumption of essential molecules (e.g., oxygen)
- Climate Regulation—Stabilization of climatic conditions
- Coastline Regulation—Stabilization of coastal lands (e.g., mangroves and reefs)
- Disease Regulation—Natural systems that reduce human disease or disease vectors
- Water Regulation—Timing and volume of water distribution across the landscape
- Waste Treatment—Filtering and treatment of waste products (incl. organics and water)
- Pollination—Distribution of pollen for the purpose of plant reproduction
**Supporting—Services that are not necessary for all other ecosystem services (Least amenable to technological replacement):**
- Soil Formation—The creation of new soil
- Nutrient Cycling—The movement of nutrients through the ecosystems
- Primary Production—The creation of sugars from sunlight
For this part:
**Decision:** Output “Y” if the abstract explicitly describes a practical technological method that contributes to one or more of these services; otherwise, output “N”.
**Category:**
If Decision is “Y”, choose **one** of the following:
- **”Support”** assists or maintains an existing natural process without intensifying it. Example: “Adding baffles so river flow still scours sediment but a little more efficiently.”
- **”Enhance”** significantly boosts the efficiency or scale of a natural process while still relying on that process. Example: “Embedding enzymes in a filter to double the nitrification rate; process still needs microbes.”
- **”Replace”** creates an artificial substitute that operates independently of the natural process. Example: “A photocatalytic panel that fixes nitrogen from air in total isolation from biological pathways.”
(If uncertain between Enhance and Replace, choose Enhance.)
Leave blank if Decision is “N”
**EcosystemService:** If Decision is “Y”, provide the exact ecosystem service from the list.
**Technology:** If Decision is “Y”, provide a concise short name for the technology used.
2. **Review Paper Detection:**
Determine if the abstract indicates that the article is a review paper. If the abstract contains phrases like “review”, “survey”, “meta-analysis”, or other similar indicators, then:
- **ReviewFlag:** Set to “review”.
Otherwise, leave this field blank.

**Output Format (Strict JSON Only)**
Your output must be in JSON format only, following this structure:
{
  "Decision": "Y" or "N",
  "Category": "Support" or "Enhance" or "Replace" (leave blank if Decision = "N"),
  "EcosystemService": "(exact ecosystem service from the list)" (leave blank if Decision = "N"),
  "Technology": "(concise short name of the technology)" (leave blank if Decision = "N"),
  "ReviewFlag": "review" or "",
  "Confidence": "high" or "medium" or "low"
}

**How to report Confidence (be calibrated, not confident):**
- "high": The paper clearly and unambiguously matches one ecosystem service, and the Replace/Enhance/Support category is obvious from the abstract.
- "medium": The classification is reasonable but not certain — for example, more than one ecosystem service could arguably apply, or the Category is a judgment call.
- "low": The paper is only vaguely related to any ecosystem service, or the abstract does not give enough evidence to be sure.

Do NOT default to "high". A calibrated distribution across all papers should include a meaningful share of "medium" and "low". Reporting "low" when uncertain is more valuable than sounding confident."""

In [16]:

def call_with_retry(client, model, system_prompt, content, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": content}
                ],
                temperature=0.1,
                max_tokens=512,  
            )
            if (response is None or response.choices is None
                    or len(response.choices) == 0
                    or response.choices[0].message.content is None):
                raise ValueError("Empty or malformed response")
            return response.choices[0].message.content, "Success"
        except Exception as e:
            wait = 2 ** attempt
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                return str(e), "Failed"

In [17]:

results = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test),
                       desc=f"Benchmarking {TARGET_MODEL}"):
    paper_id      = str(row['No. (number used only for testing stage)'])
    title_text    = str(row['Article Title'])
    keywords_text = str(row['Author Keywords'])
    abstract_text = str(row['Abstract'])

    combined_content = (f"Title: {title_text}\n"
                        f"Keywords: {keywords_text}\n"
                        f"Abstract: {abstract_text}")

    raw_output, status = call_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT, combined_content
    )
    results.append({"wos_id": paper_id, "qwen_output": raw_output, "status": status})

    time.sleep(0.6)   

df_predictions = pd.DataFrame(results)

# sanity check
print(df_predictions['status'].value_counts())

Benchmarking qwen/qwen-2.5-72b-instruct: 100%|█| 100/100 [13:12<00:00,  7.93s/i

status
Success    99
Failed      1
Name: count, dtype: int64


In [24]:
# 看失败行的错误信息是什么
failed = df_predictions[df_predictions['status'] == 'Failed']
print(failed['qwen_output'].value_counts().head())

qwen_output
Empty or malformed response    1
Name: count, dtype: int64


In [26]:
failed_ids = df_predictions[df_predictions['status'] == 'Failed']['wos_id'].tolist()
df_retry = df_test[df_test['No. (number used only for testing stage)'].astype(str)
                   .str.replace(r'\.0$','',regex=True).str.strip().isin(failed_ids)]

retry_results = []
for index, row in tqdm(df_retry.iterrows(), total=len(df_retry), desc="Retrying"):
    paper_id      = str(row['No. (number used only for testing stage)']).replace('.0','').strip()
    combined_content = (f"Title: {row['Article Title']}\n"
                        f"Keywords: {row['Author Keywords']}\n"
                        f"Abstract: {row['Abstract']}")
    raw_output, status = call_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT, combined_content
    )
    retry_results.append({"wos_id": paper_id, "qwen_output": raw_output, "status": status})
    time.sleep(1.5)  

df_retry_done = pd.DataFrame(retry_results)

df_predictions['wos_id'] = df_predictions['wos_id'].astype(str).str.replace(r'\.0$','',regex=True).str.strip()
df_predictions = df_predictions[df_predictions['status'] == 'Success']
df_predictions = pd.concat([df_predictions, df_retry_done], ignore_index=True)

print(df_predictions['status'].value_counts())

Retrying: 100%|██████████████████████████████████| 1/1 [00:05<00:00,  5.22s/it]

status
Success    100
Name: count, dtype: int64


In [27]:
# 1. JSON parser — Extracting fields from Qwen's raw output
def _parse_llm_json(output_str):
    try:
        match = re.search(r'\{.*\}', str(output_str), re.DOTALL)
        data = json.loads(match.group()) if match else json.loads(output_str)
        return pd.Series({
            'qwen_decision':   str(data.get("Decision", "")).strip(),
            'qwen_category':   str(data.get("Category", "")).strip(),
            'qwen_service':    str(data.get("EcosystemService", "")).strip(),
            'qwen_review':     str(data.get("ReviewFlag", "")).strip().lower(),
            'qwen_confidence': str(data.get("Confidence", "")).strip().lower(),  # ← 新增
        })
    except Exception:
        return pd.Series({
            'qwen_decision': None, 'qwen_category': None,
            'qwen_service': None, 'qwen_review': None,
            'qwen_confidence': None,
        })


df_predictions[['qwen_decision', 'qwen_category', 'qwen_service', 'qwen_review', 'qwen_confidence']] = \
    df_predictions['qwen_output'].apply(_parse_llm_json)

# 2.Merging prediction results with human-verified data

col_id_val = 'No. (number used only for testing stage)'

# clean IDs on both sides to ensure merge alignment (remove .0 suffix and whitespaces)
df_predictions['wos_id'] = (
    df_predictions['wos_id'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)
df_validate[col_id_val] = (
    df_validate[col_id_val].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)

# Exclude summary rows like row 101 (where human columns are all empty)
df_validate_clean = df_validate[df_validate['Decision (human)'].notna()].copy()

df_merged = pd.merge(
    df_validate_clean,
    df_predictions,
    left_on=col_id_val,
    right_on='wos_id',
    how='inner'
)

In [28]:

# 3. Define normalization function to remove false errors caused by formatting
def normalize_service(s):
    """Normalize service names by removing whitespace, casing, and delimiter differences."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r'\s*/\s*', '/', s)   # "Fibre / Hide" -> "fibre/hide"
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def normalize_simple(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return str(s).strip().lower()


# 4. Unified scoring function: pass 4 prediction column names and return the row score
# Scoring function 
def score_row(row, dec_col, cat_col, srv_col, rev_col):
    h_dec = normalize_simple(row['Decision (human)'])
    h_cat = normalize_simple(row['Category (human)'])
    h_srv = normalize_service(row['EcosystemService (human)'])
    h_rev = normalize_simple(row['ReviewFlag (human)'])

    p_dec = normalize_simple(row[dec_col])
    p_cat = normalize_simple(row[cat_col])
    p_srv = normalize_service(row[srv_col])
    p_rev = normalize_simple(row[rev_col])

   
    if 'review' in h_rev and 'review' in p_rev:
        return 1.0
        
    if h_dec != p_dec:
        return 0.0
    
    if h_dec == 'n' and p_dec == 'n':
        return 1.0
        
    if h_dec == 'y' and p_dec == 'y':
        cat_match = (h_cat == p_cat)
        srv_match = (h_srv == p_srv)
        
        if cat_match and srv_match:
            return 1.0
        elif cat_match or srv_match:
            return 0.5
        else:
            return 0.0
            
    return 0.0

In [29]:
# GPT-4.1 scoring — Verifying alignment between the scoring function and human labels
df_merged['gpt_score'] = df_merged.apply(
    lambda r: score_row(r, 'Decision (gpt)', 'Category (gpt)',
                        'EcosystemService (gpt)', 'ReviewFlag (gpt)'),
    axis=1
)
gpt_final = df_merged['gpt_score'].mean() * 100

# Baseline: Author's Human Score Mean (Excluding Summary Rows)
author_final = df_validate_clean['accuracy score '].dropna().mean() * 100

# Qwen scoring
df_merged['qwen_score'] = df_merged.apply(
    lambda r: score_row(r, 'qwen_decision', 'qwen_category',
                        'qwen_service', 'qwen_review'),
    axis=1
)
qwen_final = df_merged['qwen_score'].mean() * 100

In [30]:
# summary
print("="*60)
print("Model Benchmarking (rule-based scoring)")
print("="*60)
print(f"Author's manual score for GPT-4.1 : {author_final:5.2f}")
print(f"My rule-based score for GPT-4.1   : {gpt_final:5.2f}")
print(f"My rule-based score for Qwen      : {qwen_final:5.2f}")
print(f"GPT-4.1 vs Qwen gap               : {gpt_final - qwen_final:5.2f}")
print("="*60)

Model Benchmarking (rule-based scoring)
Author's manual score for GPT-4.1 : 84.50
My rule-based score for GPT-4.1   : 83.50
My rule-based score for Qwen      : 72.50
GPT-4.1 vs Qwen gap               : 11.00


In [31]:
# Confidence distribution
print("\nConfidence distribution:")
print(df_merged['qwen_confidence'].value_counts())

# Accuracy by confidence level
print("\nAccuracy by confidence level:")
for level in ['high', 'medium', 'low']:
    sub = df_merged[df_merged['qwen_confidence'] == level]
    if len(sub) > 0:
        acc = sub['qwen_score'].mean() * 100
        print(f"  {level:6s}: n={len(sub):3d}  accuracy={acc:.1f}")


Confidence distribution:
qwen_confidence
high      67
medium    32
low        1
Name: count, dtype: int64

Accuracy by confidence level:
  high  : n= 67  accuracy=76.9
  medium: n= 32  accuracy=62.5
  low   : n=  1  accuracy=100.0
